In [1]:
import sys
import pandas as pd
import numpy as np
import navis
import pymaid
import random
import re
import ast
import json


import skeletor as sk
import cloudvolume as cv
import fafbseg
import caveclient
import k3d

D:\flywire_backup\cave\cave\lib\site-packages\blessed\terminal.py:183: UserWarning: Failed to setupterm(kind='xterm-color'): Could not find terminal xterm-color
  warnings.warn(msg)


### auth and import

In [2]:
#megalopta project id: 8
#megalopta test project id: 25
#megalopta flywire project id: 28

path = 'D:/flywire_backup/cave/cat_client.json' # replace with your path

with open(path, 'r') as f:
    conn_file = json.load(f)
api_token = conn_file['api_token']
http_user = conn_file['http_user']
http_password = conn_file['http_password']
server = conn_file['server']

cat_client = pymaid.CatmaidInstance(server = server, 
                                api_token = api_token,
                                caching = True,
                                project_id = 8,
                                http_user = http_user,
                                http_password = http_password 
                                )

megalopta_flywire = pymaid.CatmaidInstance(server = server, 
                                api_token = api_token,
                                caching = True,
                                project_id = 28,
                                http_user = http_user,
                                http_password = http_password 
                                )


client = caveclient.CAVEclient(server_address='https://global.connectomics.braininbrain.org')

client_pb = caveclient.CAVEclient(datastack_name='megalopta_pb1_datastack')
auth_pb = client_pb.auth
cg_pb = client_pb.chunkedgraph
print('client set to PB data')

client_eb = caveclient.CAVEclient(datastack_name='megalopta_fb_eb_datastack', auth_token='34cfd1f903be508ac3068836dd10edc6')
auth_eb = client_eb.auth
cg_eb = client_eb.chunkedgraph
print('client set to EB data')


################################ other useful functions
# client.info.get_datastacks()
#print(f"My current token is: {auth.token}")

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)
INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


client set to PB data
client set to EB data


In [3]:
client_no = caveclient.CAVEclient(datastack_name='megalopta_no_r_datastack', auth_token='34cfd1f903be508ac3068836dd10edc6')
auth_no = client_no.auth
cg_no = client_no.chunkedgraph
print('client set to noduli data')


client set to noduli data


In [3]:
# Before we connect to the database we have to “monkey patch” cloudvolume such that it returns navis neurons:
# This needs to be run only once at the beginning of each session
# graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a
# graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_EB_v3
navis.patch_cloudvolume()
vol_eb = cv.CloudVolume('graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_FB_EB_v3', use_https=True, progress=False)
vol_pb = cv.CloudVolume('graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_PB1_v2a', use_https=True, progress=False)
vol_no = cv.CloudVolume('graphene://https://local.cave.braininbrain.org/segmentation/table/megalopta_NO_R_v4', use_https=True, progress=False)

INFO  : cloud-volume successfully patched! (navis)


In [4]:
# catmaid will create new ids for connectors, so this is just to make the uploader happy
# spatially overlapping connectors will be merged during upload, so no need to worry about uniqueness or duplicates

def id_generator(size: int = 7) -> str:
    """Creates a random string of digits of the specified length."""
    
    first_digit = random.choice('123456789')  # Ensures the first digit is not '0'
    remaining_digits = ''.join(random.choice('0123456789') for _ in range(size - 1))
    
    return first_digit + remaining_digits

print(id_generator())

resolution_xyz = np.unique([list(r.values()) for r in megalopta_flywire.image_stacks.resolution], axis=0).min(0).astype(int)

4198018


In [8]:
# associate pre/post L1 ids and updated roots with google sheet neurons (make sure to update root ids for those)

flywire_progress = pd.read_csv('./Eciton_FB neuron_CAVE progress - eciton_FB_PFN.csv', dtype=str)
n_table = flywire_progress #[flywire_progress['Catmaid name'].str.contains(neurons, na=False, regex=True)]

#select only relevant columns and rename for ease of use
nametable = n_table[['Catmaid name', 'Root ID', 'CATMAID skid']].rename(columns={'Catmaid name':'neuron_name', 'Root ID':'root_ids', 'CATMAID skid':'skid'})


nametable = nametable.dropna(subset='skid')

# nametable['skid'] = nametable['skid'].astype(int)
nametable['skid'] = nametable['skid'].fillna(0).astype(float).astype(int)

# probably a better way to do this...
# convert each cell in 'root_ids' from a comma-separated string to a list of integers, ignoring empty strings
# UNCOMMENT BELOW FOR MULTI ROOT
# nametable['root_ids'] = nametable['root_ids'].apply(lambda x: [int(seg) for seg in x.split(',') if seg.strip()])

nametable['root_ids'] = nametable['root_ids'].astype(str)

# Convert string representations of lists to actual lists
nametable['root_ids'] = nametable['root_ids'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else x)

# Remove brackets by joining list elements into a string
nametable['root_ids'] = nametable['root_ids'].apply(lambda x: ', '.join(map(str, x)) if isinstance(x, list) else str(x))

nametable

,neuron_name,root_ids,skid
13,PFN_LY_L4_AT_MS_VN2_PFN2E,504403158739739636,61663
23,PFN_LY_L4_MS_VN3_PFN2E,504403158705447288,123278
24,PFN_LY_L4_MS_VN4_PFN2E,504403158710540307,123288
25,PFN_LY_L4_MS_VN5_PFN3E,504403158928391356,123298
26,PFN_LY_L4_MS_VN6__PFN4E_PFNc-maybe,504403158747624712,123325
27,PFN_LY_L4-maybe_MS_VN7_PFN5E,504403158700493608,123365
28,PFN_LY_L4_MS_VN8_PFN5E,504403158803500127,123506
29,PFN_LY_L4_MS_VN9_PFN4E,504403158688922673,123856
30,PFN_LY_L4-likely_MS_PFN5E_PFNc-maybe,504403158603543395,123863
31,PFN_LY_L4-likely_MS_VN10_PFN4E_PFNc-maybe,504403158695419734,123872


In [10]:
# LNOs_a: 504403158459506101,
# LNOs_b: 504403158467123201, 
# LNOs_c: 504403158551966913
lno_list = [504403158459506101, 504403158467123201, 504403158551966913]
m = vol_no.mesh.get(lno_list, remove_duplicate_vertices=True, as_navis=True)

In [ ]:
'TN/LNO6_LNOs_a'
'TN/LNO4_LNOs_b'
'TN/LNO9_LNOs_c'

In [8]:
nametable = nametable.iloc[14:]

In [10]:
# separate nametable into two df: one with only single segids and another with lists of segids

# USER INPUT: set cloudvolume server
vol = vol_eb

#################################

oneRoot = nametable[~nametable['root_ids'].str.contains(',')] # check that string doesn't contain comma
multiRoot = nametable[nametable['root_ids'].str.contains(',')] # some roots are not merged due to NoneType error

oneRoot_list = oneRoot['root_ids'].tolist()
multiRoot_list = multiRoot['root_ids'].tolist()

# download neurons that have single root id
oneRoot_m = []

for idx, neuron in enumerate(oneRoot_list, start = 1):
    m = vol.mesh.get(neuron, remove_duplicate_vertices=True, as_navis=True)
    oneRoot_m.append(m)
    sys.stdout.write(f'\r{idx}/{len(oneRoot_list)} neuron mesh with single root downloaded.') # update console with progress
    sys.stdout.flush()

# download and fuse neurons that have multiple unmerged root ids

multiRoot_id_list = []
multiRoot_m = [] # mesh list

for idx, neuron in enumerate(multiRoot_list, start=1):
    n = list(neuron.strip().split(','))
    firstRoot = n[0]# multiRoot_list[0].split(',')[0].strip()
    multiRoot_id_list.append(firstRoot)
    m = vol.mesh.get(n, remove_duplicate_vertices=True, fuse=True, allow_missing=True, as_navis=True)
    multiRoot_m.append(m)
   
    sys.stdout.write(f'\r{idx}/{len(multiRoot_list)} neuron mesh with multi root ids downloaded and fused.') # update console with progress
    sys.stdout.flush()

oneRoot_m = navis.NeuronList(oneRoot_m)
multiRoot_m = navis.NeuronList(multiRoot_m)
multiRoot_m.set_neuron_attributes(multiRoot_id_list, name='id')

# merge oneRoot_m and multiRoot_m meshNeuronLists

# Check if both exist in the global namespace
if 'oneRoot_m' in globals() and 'multiRoot_m' in globals():
    m = oneRoot_m + multiRoot_m
    print(f'Combined list contains {len(m)} mesh neurons.')
elif 'oneRoot_m' in globals():
    m = oneRoot_m
    print(f'Combined list contains {len(m)} mesh neurons.')
elif 'multiRoot_m' in globals():
    m = multiRoot_m
    print(f'Combined list contains {len(m)} mesh neurons.')
else:
    print('uh-oh - major epic yikes sauce!')


4/4 neuron mesh with single root downloaded.Combined list contains 4 mesh neurons.


### fig 1 representative neurons

In [7]:
# epg_l3
m_eb = vol_eb.mesh.get(576460753102309773, remove_duplicate_vertices=True, as_navis=True)
m_pb = vol_pb.mesh.get([576460752504703640,576460752494001689], fuse=True, remove_duplicate_vertices=True, as_navis=True)
m = m_eb + m_pb

# peg_L2
m_eb = vol_eb.mesh.get(576460752959971604, remove_duplicate_vertices=True, as_navis=True)
m_pb = vol_pb.mesh.get([576460752482567425,576460752597474821], fuse=True, remove_duplicate_vertices=True, as_navis=True)
m = m_eb + m_pb

# pen_l3
m_eb = vol_eb.mesh.get(576460753130945957, remove_duplicate_vertices=True, as_navis=True)
m_pb = vol_pb.mesh.get([576460752591088521], fuse=True, remove_duplicate_vertices=True, as_navis=True)


In [ ]:
skel = m.copy() # just in case

# # for neuron in skel:
# #     root_id = str(neuron.id)
# #     sk_row = nametable.loc[nametable['root_ids'].str.contains(root_id)]
# #     name = sk_row['neuron_name'].values[0] if not sk_row.empty else None
# #     skid = sk_row['skid'].values[0] if not sk_row.empty else None
# #     skid = str(int(skid))
    
# #     if not sk_row.empty:
# #         neuron.name = name + '_' + skid
# #         print(neuron.name)
        
# simplify
m_simp = navis.simplify_mesh(skel, 0.3, backend='pyfqmr', inplace=False, parallel=True, progress=True) #lower value, more simple


skel_list = []
# skeletonize
for neuron in m_simp:
    skel = fafbseg.flywire.skeletonize_neuron(neuron) 
    skel_list.append(skel)

skel_list = navis.NeuronList(skel_list)

# then join aka 'heal' unjoined fragments
sk_h_list = []

for idx, neuron in enumerate(skel_list, start=1):
    if len(neuron.root) > 1: # number of disconnected fragments
        skel_h = navis.heal_skeleton(neuron)
        skel_h.soma = None
        sk_h_list.append(skel_h)
        print(f'\r{idx}/{len(skel_list)} skeleton healed')
        sys.stdout.flush()

sk_h_list = navis.NeuronList(sk_h_list)

sk_h_list

In [ ]:
fig=navis.plot3d([sk_h_list[0], sk_h_list[1]], radius=True)

In [16]:
for idx, neuron in enumerate(sk_h_list, start=1):
    neuron.nodes['radius'] = neuron.nodes['radius'].replace(0, 50) # smallest node size shouldn't be 0. 50 is arbitrary

# for idx, neuron in enumerate(sk_h_list, start=1):
#     count += 1
#     neuron.name = f"EPG_L3_{count}"

for idx, neuron in enumerate(sk_h_list, start=1):
    neuron.name = "EPG_LZ_L3_ID_SH"

# downsample neurons to make upload to catmaid quicker
skel_ds = []

for neuron in sk_h_list:
    n_ds = navis.downsample_neuron(neuron, downsampling_factor=1.5, inplace=False)
    skel_ds.append(n_ds)
    

In [ ]:
fig=navis.plot3d([skel_ds[0], skel_ds[1]], radius=True)

In [21]:
# for neuron in skel_ds:
#     pymaid.upload_neuron(neuron, import_connectors=False, remote_instance=megalopta_flywire)

for neuron in sk_h_list:
    pymaid.upload_neuron(neuron, import_connectors=False, remote_instance=megalopta_flywire)

### full neuron list upload

In [10]:
m

,type,name,id,units,n_vertices,n_faces
0,navis.MeshNeuron,None,576460752943174870,1 nanometer,487460,971865
1,navis.MeshNeuron,None,576460753001015108,1 nanometer,399522,800004
...,...,...,...,...,...,...
16,navis.MeshNeuron,None,576460753031772744,1 nanometer,840699,1681072
17,navis.MeshNeuron,None,576460753128778871,1 nanometer,527049,1055452


In [9]:
m.name 

array([None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None, None,
       None, None, None, None, None, None, None, None, None, None],
      dtype=object)

In [11]:
skel = m.copy() 

for neuron in skel:
    root_id = str(neuron.id)
    sk_row = nametable.loc[nametable['root_ids'].str.contains(root_id)]
    name = sk_row['neuron_name'].values[0] if not sk_row.empty else None
    skid = sk_row['skid'].values[0] if not sk_row.empty else None
    skid = str(int(skid))
    
    if not sk_row.empty:
        neuron.name = name + '_' + skid 
        print(neuron.name)
        

ER_TL_L_MBUv_SH_ES_14_125261
ER_TL_L_MBUv_SH_ES_10_125025
ER_TL_L_MBUv_SH_ES_18_125035
ER_TL_L_MBUv_SH_ES_19_125041
ER_TL_L_MBUv_SH_ES_21_125099
ER_TL_L_MBUv_SH_ES_23_125194
ER_TL_L_MBUv_SH_ES_24_125199
ER_TL_L_MBUv_SH_ES_27_125291
ER_TL_R_LAL_SH_MS_47232
ER_TL_R_LAL_SH_MS_57041
ER_TL_R_MBUd_ES_26_SH_compl_syn_184355
ER_TL_R_MBUd_ES_42_syn_189306
ER_TL_R_MBUd_SH_MS_62_63046
ER_TL_R_MBUv_01_63126
ER_TL_R_MBUv_02_64133
ER_TL_R_MBUv_SH_ES_21_47185
ER_TL_L_MBUv_SH_ES_22_125189
ER_TL_L_MBUv_SH_ES_25_125250


In [19]:
skel

,
type,navis.TreeNeuron
name,None
id,576460752943174870
n_nodes,9205
n_connectors,None
n_branches,306
n_leafs,420
cable_length,2756174.380258
soma,10603
units,1 nanometer


In [21]:
#### SIMPLIFY MESH ####
skel = m.copy()
m_simp = navis.simplify_mesh(skel, 0.3, backend='pyfqmr', inplace=False, parallel=True) #lower value, more simple

Simplifying:   0%|          | 0/18 [00:00<?, ?it/s]

In [22]:
skel_list = []
# skeletonize
for neuron in m_simp:
    skel = fafbseg.flywire.skeletonize_neuron(neuron, dataset=None) 
    skel_list.append(skel)

skel_list = navis.NeuronList(skel_list)

Skeletonizing:   0%|          | 0/141061 [00:00<?, ?it/s]

Skeletonizing:   0%|          | 0/116152 [00:00<?, ?it/s]

Skeletonizing:   0%|          | 0/239033 [00:00<?, ?it/s]

Skeletonizing:   0%|          | 0/152876 [00:00<?, ?it/s]

Skeletonizing:   0%|          | 0/73517 [00:00<?, ?it/s]

...

Skeletonizing:   0%|          | 0/115453 [00:00<?, ?it/s]

Skeletonizing:   0%|          | 0/170673 [00:00<?, ?it/s]

Skeletonizing:   0%|          | 0/258406 [00:00<?, ?it/s]

Skeletonizing:   0%|          | 0/242118 [00:00<?, ?it/s]

Skeletonizing:   0%|          | 0/153283 [00:00<?, ?it/s]

In [48]:
# if fafbseg.flywire.skeletonize_neuron doesn't work
# import skeletor as sk

# skel_list = []
# # skeletonize
# for neuron in m_simp:
#     fixed = sk.pre.fix_mesh(neuron, remove_disconnected=5, inplace=False)
#     skel = sk.skeletonize.by_wavefront(fixed, waves=1, step_size=1)
#     skel_list.append(skel)

# skel_list = navis.NeuronList(skel_list)

Skeletonizing:   0%|          | 0/246157 [00:00<?, ?it/s]

In [23]:
# then join aka 'heal' unjoined fragments
sk_h_list = []

for idx, neuron in enumerate(skel_list, start=1):
    if len(neuron.root) > 1: # number of disconnected fragments
        skel_h = navis.heal_skeleton(neuron)
        skel_h.soma = None
        sk_h_list.append(skel_h)
        print(f'\r{idx}/{len(skel_list)} skeleton healed')
        sys.stdout.flush()

sk_h_list = navis.NeuronList(sk_h_list)

for idx, neuron in enumerate(sk_h_list, start=1):
    neuron.nodes['radius'] = neuron.nodes['radius'].replace(0, 50) # smallest node size shouldn't be 0. 50 is arbitrary

sk_h_list

1/18 skeleton healed
2/18 skeleton healed
3/18 skeleton healed
4/18 skeleton healed
5/18 skeleton healed
6/18 skeleton healed
7/18 skeleton healed
8/18 skeleton healed
9/18 skeleton healed
10/18 skeleton healed
11/18 skeleton healed
12/18 skeleton healed
13/18 skeleton healed
14/18 skeleton healed
15/18 skeleton healed
16/18 skeleton healed
17/18 skeleton healed
18/18 skeleton healed


,type,name,id,n_nodes,n_connectors,n_branches,n_leafs,cable_length,soma,units
0,navis.TreeNeuron,None,576460752943174870,9205,None,383,406,2.793804e+06,None,1 nanometer
1,navis.TreeNeuron,None,576460753001015108,7000,None,334,354,2.098583e+06,None,1 nanometer
...,...,...,...,...,...,...,...,...,...,...
16,navis.TreeNeuron,None,576460753031772744,13517,None,645,691,4.210482e+06,None,1 nanometer
17,navis.TreeNeuron,None,576460753128778871,8623,None,425,463,2.652698e+06,None,1 nanometer


In [ ]:
# check skeletonization results (can use radius=True)
navis.clear3d()
fig=navis.plot3d(skel_list,radius=True)

In [59]:
names = pymaid.get_skids_by_name('/.*glom.*', remote_instance=cat_client)

In [61]:

glom = pymaid.get_neuron(names['skeleton_id'], with_connectors=False, remote_instance=cat_client)

Fetch neurons:   0%|          | 0/176 [00:00<?, ?it/s]

Make nrn:   0%|          | 0/176 [00:00<?, ?it/s]

In [65]:
skid = 184355
er = pymaid.get_neuron(skid, with_connectors=False, remote_instance=cat_client)

In [24]:
name = name.replace("/", "_")

name

'TN_LNO8_todo_AT_SH'

In [24]:
# may need to rename neurons again, not sure why

for neuron in sk_h_list:
    root_id = str(neuron.id)
    sk_row = nametable.loc[nametable['root_ids'].str.contains(root_id)]
    name = sk_row['neuron_name'].values[0] if not sk_row.empty else None
    name = name.replace("/", "_")
    skid = sk_row['skid'].values[0] if not sk_row.empty else None
    skid = str(int(skid))
    
    if not sk_row.empty:
        neuron.name = name + '_' + skid
        print(neuron.name)

sk_h_list

ER_TL_L_MBUv_SH_ES_14_125261
ER_TL_L_MBUv_SH_ES_10_125025
ER_TL_L_MBUv_SH_ES_18_125035
ER_TL_L_MBUv_SH_ES_19_125041
ER_TL_L_MBUv_SH_ES_21_125099
ER_TL_L_MBUv_SH_ES_23_125194
ER_TL_L_MBUv_SH_ES_24_125199
ER_TL_L_MBUv_SH_ES_27_125291
ER_TL_R_LAL_SH_MS_47232
ER_TL_R_LAL_SH_MS_57041
ER_TL_R_MBUd_ES_26_SH_compl_syn_184355
ER_TL_R_MBUd_ES_42_syn_189306
ER_TL_R_MBUd_SH_MS_62_63046
ER_TL_R_MBUv_01_63126
ER_TL_R_MBUv_02_64133
ER_TL_R_MBUv_SH_ES_21_47185
ER_TL_L_MBUv_SH_ES_22_125189
ER_TL_L_MBUv_SH_ES_25_125250


,type,name,id,n_nodes,n_connectors,n_branches,n_leafs,cable_length,soma,units
0,navis.TreeNeuron,ER_TL_L_MBUv_SH_ES_14_125261,576460752943174870,9205,None,383,406,2.793804e+06,None,1 nanometer
1,navis.TreeNeuron,ER_TL_L_MBUv_SH_ES_10_125025,576460753001015108,7000,None,334,354,2.098583e+06,None,1 nanometer
...,...,...,...,...,...,...,...,...,...,...
16,navis.TreeNeuron,ER_TL_L_MBUv_SH_ES_22_125189,576460753031772744,13517,None,645,691,4.210482e+06,None,1 nanometer
17,navis.TreeNeuron,ER_TL_L_MBUv_SH_ES_25_125250,576460753128778871,8623,None,425,463,2.652698e+06,None,1 nanometer


In [25]:
pymaid.upload_neuron(sk_h_list, import_connectors=False, remote_instance=megalopta_flywire)

Uploading:   0%|          | 0/18 [00:00<?, ?it/s]

{576460752943174870: {'neuron_id': 463717,
  'skeleton_id': 463716,
  'node_id_map': {1: 8396541,
   10243: 8396542,
   9609: 8396543,
   8243: 8396545,
   7550: 8396547,
   7169: 8396549,
   6960: 8396551,
   6933: 8396553,
   6923: 8396555,
   4631: 8396557,
   4630: 8396559,
   10245: 8396560,
   6153: 8396562,
   5580: 8396564,
   9685: 8396566,
   9684: 8396568,
   5139: 8396569,
   3972: 8396571,
   4634: 8396573,
   4643: 8396575,
   4135: 8396577,
   1871: 8396579,
   3622: 8396581,
   2098: 8396583,
   2091: 8396585,
   2351: 8396587,
   2861: 8396589,
   3265: 8396591,
   3326: 8396593,
   2507: 8396595,
   1818: 8396597,
   1911: 8396599,
   1604: 8396601,
   1307: 8396603,
   1308: 8396605,
   3615: 8396606,
   810: 8396608,
   647: 8396610,
   578: 8396612,
   1077: 8396614,
   36: 8396616,
   70: 8396617,
   71: 8396618,
   73: 8396620,
   74: 8396621,
   72: 8396619,
   76: 8396623,
   93: 8396625,
   96: 8396627,
   79: 8396626,
   80: 8396628,
   130: 8396630,
   77: 8

### figure plots

In [20]:
sk_h_list

,type,name,id,n_nodes,n_connectors,n_branches,n_leafs,cable_length,soma,units
0,navis.TreeNeuron,TN_LNO1_R_SH_AT_47215_NO,504403158488287832,32335,None,2627,2933,9.376130e+06,None,1 nanometer
1,navis.TreeNeuron,TN_LNO2_R_SH_AT_47221_NO,504403158401056605,37817,None,3095,3440,1.135861e+07,None,1 nanometer
...,...,...,...,...,...,...,...,...,...,...
4,navis.TreeNeuron,TN/LNO7_todo_AT_SH_57076_NO,504403158402520236,13481,None,1342,1609,4.480783e+06,None,1 nanometer
5,navis.TreeNeuron,TN/LNO8_todo_AT_SH_57081_NO,504403158268845171,9274,None,1046,1304,3.220265e+06,None,1 nanometer


In [27]:
# save meshes locally...
# navis.write_mesh(sk_h_list, './hd_neurons/raw_neuron_meshes/NO/{neuron.name}.obj', filetype='obj')
navis.write_swc(sk_h_list, './hd_neurons/raw_neuron_meshes/NO/{neuron.name}.swc')

Writing:   0%|          | 0/6 [00:00<?, ?it/s]

In [20]:
m = vol_eb.mesh.get(EPG_L3, remove_duplicate_vertices=True, as_navis=True)

for neuron in m:
    root_id = str(neuron.id)
    sk_row = nametable.loc[nametable['root_ids'].str.contains(root_id)]
    name = sk_row['neuron_name'].values[0] if not sk_row.empty else None
    skid = sk_row['skid'].values[0] if not sk_row.empty else None
    skid = str(int(skid))
    
    if not sk_row.empty:
        neuron.name = name + '_' + skid + '_' + 'EB'
        print(neuron.name)

navis.write_mesh(m, './hd_neurons/raw_neuron_meshes/EB/{neuron.name}.obj', filetype='obj')

EPG_LZ_L3_ID_SH_54939_EB


Writing:   0%|          | 0/1 [00:00<?, ?it/s]

In [21]:
m

,type,name,id,units,n_vertices,n_faces
0,navis.MeshNeuron,EPG_LZ_L3_ID_SH_54939_EB,576460753102309773,1 nanometer,1098485,2196365


In [23]:
skid = 54939
cat_list = pymaid.get_neuron(skid, with_connectors=False, remote_instance=cat_client)

cat_n = cat_list.copy()
match = re.search(r'^([A-Z]{3}).*?([LR]\d)', cat_n.name)
if match:
    extracted = f"{match.group(1)}_{match.group(2)}"
    cat_n.name = extracted + '_' + cat_n.id + '_' + 'backbone'
    print(cat_n.name)
else:
    print(f"Original: {cat_n.name} → No match found")


cat_n

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)
INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


EPG_L3_54939_backbone


,
type,CatmaidNeuron
name,EPG_L3_54939_backbone
id,54939
n_nodes,1425
n_connectors,0
n_branches,80
n_leafs,86
cable_length,1243720.25
soma,None
units,1 nanometer


In [40]:
#### SIMPLIFY MESH ####
m_simp = navis.simplify_mesh(sk_h_list, 0.5, backend='pyfqmr', inplace=False, parallel=True) #lower value, more simple

Simplifying:   0%|          | 0/19 [00:00<?, ?it/s]

In [41]:
# save meshes locally...
navis.write_mesh(m_simp, './hd_neurons/simplified_neuron_meshes/{neuron.name}.obj', filetype='obj')

Writing:   0%|          | 0/19 [00:00<?, ?it/s]

In [85]:
 # When reading all files in folder you have to specificy the file extension (e.g. *.stl)
 meshes = navis.read_mesh('./hd_neurons/simplified_neuron_meshes/*.obj')

Importing:   0%|          | 0/19 [00:00<?, ?it/s]

In [ ]:
# Load a mesh file into a Volume
# vol = navis.read_mesh('test_mesh.stl', output='volume')

In [10]:
# get EB and PB volumes from Megalopta dataset 
eb_m = pymaid.get_volume(['EB'], remote_instance=cat_client)
pb_m = pymaid.get_volume(['PB_new'], remote_instance=cat_client)

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


In [60]:
eb_m.color = (0, 0, 0, .02)
pb_m.color = (0, 0, 0, .02)

In [64]:
skid_list = nametable['skid'].to_list()

cat_list = []

for skid in skid_list:
    cat_n = pymaid.get_neuron(skid, with_connectors=False, remote_instance=cat_client)
    cat_list.append(cat_n)

cat_list = navis.NeuronList(cat_list)

cat_list

,type,name,id,n_nodes,n_connectors,n_branches,n_leafs,cable_length,soma,units
0,CatmaidNeuron,EPG_LZ_L2_2_SH_HH_MS,55944,19441,0,1478,1547,7088621.0,None,1 nanometer
1,CatmaidNeuron,EPG_LZ_L2_3_SH_MS_HH,55952,16741,0,1141,1190,6062857.0,None,1 nanometer
...,...,...,...,...,...,...,...,...,...,...
21,CatmaidNeuron,PEN_RW_R9_SH_MT5,64257,4557,0,273,288,2240929.0,None,1 nanometer
22,CatmaidNeuron,PEN_LY_L4_SH_SB_cina,99084,2529,0,163,172,1551647.5,None,1 nanometer


In [96]:
EPG_R1_sk = cat_list[10:13]

In [ ]:
# Convert to Pandas Series to use .str.contains()
EPG_R1 = pd.Series(meshes.name).str.contains('R1')
EPG_R1 = meshes[EPG_R1]
EPG_R1


In [102]:
EPG_R1_sk.color = (0.1, 0.1, 0.1)
# color={EPG_R1_sk[0]: (65, 65, b), ...}

In [101]:
EPG_R1_sk.color

(65, 65, 65)

In [ ]:
EPG_R1_sk

In [ ]:
#to do: set neuron-color dict using navis default colors

In [108]:
navis.clear3d()
navis.plot3d([[EPG_R1[0], EPG_R1_sk[0], eb_m]], color={EPG_R1_sk[0]: (0.1,0.1,0.1)}, backend='k3d', inline=False)



INFO  : Use the `.display()` method to show the plot. (navis)


Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [ ]:
# Clear previous plots
navis.clear3d()

# Step 1: Adjust the opacity of the brain volume.
# Increase opacity from 0.02 to 0.2 for visibility.
eb_m.color = (0, 0, 0, 0.03)

# Step 2: Plot the neurons and the brain volume using the plotly backend.
# Note: We use a nested list to group objects. Adjust this structure if needed.
fig = navis.plot3d([eb_m], backend='plotly', inline=False)

# Step 3: Update the figure layout:
# - Set an explicit title.
# - Update the camera to use an orthographic projection and control orientation.
fig.update_layout(
    title='example',
    scene_camera=dict(
        projection=dict(type='orthographic'),
        eye=dict(x=0.1, y=0.1, z=1.25)
    )
)

for trace in fig.data:
    # Option 1: If the trace name contains 'eb_m', adjust its opacity.
    if hasattr(trace, 'name') and trace.name is not None and 'eb_m' in trace.name:
        trace.opacity = 0.03  # adjust this value as needed

# Step 4: Render the figure.
fig.show()



In [ ]:
# Helper function: Convert navis (r, g, b, a) color tuple to 24-bit hex integer.
def convert_color(color_tuple):
    r, g, b, a = color_tuple
    # Scale values if they are normalized
    if r <= 1.0 and g <= 1.0 and b <= 1.0:
        r = int(r * 255)
        g = int(g * 255)
        b = int(b * 255)
    else:
        r = int(r)
        g = int(g)
        b = int(b)
    return (r << 16) + (g << 8) + b

# Create the k3d plot without width/height arguments.
plot = k3d.plot(camera_auto_fit=False)

# Manually set the camera.
# The camera expects a list of 9 numbers:
# [pos_x, pos_y, pos_z, center_x, center_y, center_z, up_x, up_y, up_z]
# Adjust these values for your desired orientation and zoom.
plot.camera = [0, 0, 300,    # Camera position: 300 units along z-axis
               0, 0, 0,      # Look-at point: the origin
               0, 1, 0]      # Up vector: positive y-axis

# Add neuron meshes to the plot.
# 'meshes' is a navis.NeuronList containing MeshNeuron objects.
for neuron in meshes:
    vertices = neuron.vertices.astype(np.float32)
    faces = neuron.faces.astype(np.uint32)
    
    # If neuron has a color attribute, use it; otherwise default to gray.
    if hasattr(neuron, 'color') and neuron.color is not None:
        color_hex = convert_color(neuron.color)
        opacity = neuron.color[3] if len(neuron.color) >= 4 else 1.0
    else:
        color_hex = 0xAAAAAA
        opacity = 1.0
    
    neuron_mesh = k3d.mesh(vertices, faces, color=color_hex, opacity=opacity)
    plot += neuron_mesh

# Add the brain volume.
# 'eb_m' is a navis.Volume object.
vol_vertices = eb_m.vertices.astype(np.float32)
vol_faces = eb_m.faces.astype(np.uint32)
vol_color = convert_color(eb_m.color)
vol_opacity = 0.05  # Set the desired transparency

volume_mesh = k3d.mesh(vol_vertices, vol_faces, color=vol_color, opacity=vol_opacity)
plot += volume_mesh

# Optionally display the interactive plot to verify the orientation.
plot.display()

# To generate a static 2D screenshot, capture the current view.
#screenshot = plot.get_screenshot()

# Save the screenshot as a PNG file.
#with open('static_view.png', 'wb') as f:
#    f.write(screenshot)



### random

In [29]:
sk_h_list

,type,name,id,n_nodes,n_connectors,n_branches,n_leafs,cable_length,soma,units
0,navis.TreeNeuron,EPG_RZ_R1/L1_ID_MS_MT1_64077_EB,576460752963411671,18523,None,1219,1304,4.824662e+06,None,1 nanometer
1,navis.TreeNeuron,EPG_RZ_R1/L1_ID_MT3_64084_EB,576460753015406203,11202,None,800,861,3.042089e+06,None,1 nanometer


In [22]:
navis.write_precomputed(sk_h_list, filepath='./eb_lateral_skeletons/', radius=True)

Writing:   0%|          | 0/19 [00:00<?, ?it/s]

In [28]:
sk_h_list[7].name = 'EPG_RW_R9_MS_EPGt_118347'

In [30]:
navis.write_swc(sk_h_list, './eb_lateral_skeletons/eb_skel.zip')

Writing:   0%|          | 0/19 [00:00<?, ?it/s]

In [6]:
sk_h_list = navis.read_swc('D:/analysis/eb_lateral_skeletons/eb_swc', include_subdirs=True)

Importing:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
pymaid.upload_neuron(sk_h_list, import_connectors=False, remote_instance=megalopta_flywire)

In [11]:
sk_h_list.name

array(['EPG_LZ_L2_1_PEG_ID_SH_MS_HH_55930', 'EPG_LZ_L2_2_SH_HH_MS_55944',
       'EPG_LZ_L2_3_SH_MS_HH_55952', 'EPG_RW_R8_ID_HH_56351',
       'EPG_RW_R8_ID_HH_56356', 'EPG_RW_R8_todo_SH_HH_55936',
       'EPG_RW_R9_ID_SH_64276', 'EPG_RW_R9_MS_EPGt_118347'], dtype='<U33')